In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text('vehicles',defaultValue='vehicles')
vehicles=dbutils.widgets.get('vehicles')

In [0]:
vehicles_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("battery_capacity_kwh", StringType(), True),   # quoted in JSON
    StructField("range_km", IntegerType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("registration_state", StringType(), True),
    StructField("partner_id", StringType(), True),
    StructField("effective_from", DateType(), True),
    StructField("effective_to", DateType(), True),
    StructField("is_current", BooleanType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])

vehicles_overall_schema = StructType([
    StructField("data", ArrayType(vehicles_schema), True),
    StructField("ingestion_time", StringType(), True),
    StructField("batch_no", StringType(), True),
])

In [0]:
df = (
    spark.readStream.
    format("cloudFiles").schema(vehicles_overall_schema)
    .option("cloudFiles.format", "json")
    .option("cloudFiles.maxFilesPerTrigger", "1")  # test only
    .option("multiLine", "true")
    # .option("rescuedDataColumn", "_rescued_data")
    .load(f"abfss://raw@voltgridstorageaccount.dfs.core.windows.net/{vehicles}/")
)

In [0]:
bronze_df=df.select(F.explode('data')\
    .alias('data'),'ingestion_time','batch_no')\
        .select('data.*','ingestion_time','batch_no')

In [0]:
bronze_df=bronze_df.withColumn('created_at', F.col('created_at').try_cast('date'))\
    .withColumn('updated_at', F.col('updated_at').try_cast('date'))\
        .withColumn('ingestion_time',F.col('ingestion_time').try_cast('timestamp'))\
            .withColumn('id',F.col('id').cast('int'))\
                .withColumn('batch_no',F.col('batch_no').cast('int'))\
                    .withColumn('source_path',F.col('_metadata.file_path'))\
                        .withColumn('effective_to',F.col('effective_to').cast('date'))

In [0]:
q = (bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation",
                f"abfss://controls@voltgridstorageaccount.dfs.core.windows.net/checkpoints/{vehicles}/")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"voltgrid.bronze.{vehicles}")
)
q.awaitTermination()